# ♻️ WasteWise: обучаем модель на 7 типов отходов

Этот ноутбук дообучает модель сайта (сейчас она знает 4 типа: стекло, металл, бумагу, пластик) — теперь на **7**:
+ **батарейки и электроника**, **органика**, **прочее (несортируемое)**. И учит её на обычных фото, а не только на белом фоне.

**Что понадобится:** аккаунт Google. Всё бесплатно. Время — около 1,5 часа (из них ~1 час модель учится сама).

**Как запускать:** по очереди нажимайте ▶ слева от каждой ячейки (или `Ctrl+Enter`) и ждите, пока закончится предыдущая.
Подробная инструкция — `docs/TRAINING.md` в репозитории.

## 1. Включить видеокарту (GPU)
Меню **Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель: T4 GPU → Сохранить**.
Потом запустите ячейку ниже: должна появиться строка с `Tesla T4`.

In [ ]:
!nvidia-smi -L || echo "❌ Видеокарта не включена — см. шаг 1"

## 2. Скачать код проекта
Если изменения с 7 классами ещё не влиты в `main`, впишите в `BRANCH` имя ветки из pull request.

In [ ]:
REPO = "https://github.com/adekaa11/wastewise.git"
BRANCH = "main"

!rm -rf /content/wastewise && git clone -q --depth 1 -b {BRANCH} {REPO} /content/wastewise
%cd /content/wastewise
!pip install -q "ultralytics>=8.3" "opencv-python>=4.6,<5"
!ls train

## 3. Скачать открытые датасеты
Источники и лицензии (подробно — в `train/sources.py`, авторство каждого фото попадёт в `ATTRIBUTION.csv`).
Модель, обученная на этих данных, — **только для учебного некоммерческого использования** (RealWaste — CC BY-NC-SA).

In [ ]:
from IPython.display import Markdown
from train.sources import license_table
Markdown(license_table())

In [ ]:
# TrashNet (MIT) — ~40 МБ, секунды
!mkdir -p raw/trashnet && wget -q -O /tmp/trashnet.zip https://raw.githubusercontent.com/garythung/trashnet/master/data/dataset-resized.zip && unzip -q -o /tmp/trashnet.zip -d raw/trashnet && echo "✅ TrashNet"

# RealWaste (CC BY-NC-SA 4.0) — ~650 МБ, 1–2 минуты
!git clone -q --depth 1 https://github.com/sam-single/realwaste raw/realwaste && echo "✅ RealWaste"

In [ ]:
# TACO (мусор «в природе») — фото с Flickr, 5–10 минут
!python -m train.taco --out raw/taco

In [ ]:
# Open Images — электроника (телефоны, ноутбуки, пульты, лампочки…), 3–5 минут
!python -m train.openimages --out raw/openimages --per-class 200

### 3а. (по желанию) Garbage Dataset — главный источник фото батареек
Нужен бесплатный аккаунт Kaggle: **kaggle.com → Settings → API → Create New Token** (скачается `kaggle.json`
с полями `username` и `key`). В Colab нажмите значок 🔑 **«Секреты»** слева и добавьте два секрета:
`KAGGLE_USERNAME` и `KAGGLE_KEY`, включите им «Доступ к блокноту». Затем поставьте `USE_GARBAGE_DATASET = True`.

Без этого шага батареек в обучении будет мало (их почти нет в других открытых датасетах) — модель будет
узнавать в основном электронику. Помогут и ваши фото: отправляйте с сайта «Модель ошиблась → Батарейки и электроника».

In [ ]:
USE_GARBAGE_DATASET = False

if USE_GARBAGE_DATASET:
    import os, subprocess
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "sumn2u/garbage-classification-v2",
                    "-p", "raw/garbage_dataset", "--unzip"], check=True)
    print("✅ Garbage Dataset")

## 4. Наши фото «Модель ошиблась» из Supabase
Это самое ценное: настоящие фото с телефонов. Нужны два секрета (значок 🔑 слева → «Добавить секрет»,
включить «Доступ к блокноту»):
- `SUPABASE_URL` — `https://<id проекта>.supabase.co`;
- `SUPABASE_SERVICE_KEY` — **секретный** ключ (тот же, что в Secrets сайта на Streamlit Cloud).

Ключ не вписывайте в текст ноутбука — только в «Секреты». Если секретов нет, ячейка просто пропустит этот шаг.

In [ ]:
import os, subprocess
from google.colab import userdata

try:
    os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
    os.environ["SUPABASE_SERVICE_KEY"] = userdata.get("SUPABASE_SERVICE_KEY")
except Exception:
    print("Секретов Supabase нет — учимся только на открытых датасетах")
else:
    subprocess.run(["python", "-m", "train.feedback_export", "--out", "raw/feedback"], check=True)

## 5. Собрать датасет
Фото раскладываются по 7 классам и делятся на train / val / test. **Test — одинаковый для старой и новой модели:**
фото попадает в него по своему «отпечатку» и не меняет часть при добавлении новых данных.

In [ ]:
!python -m train.dataset --raw raw --out datasets/wastewise

## 6. Обучение (~1 час)
Начинаем с текущей модели сайта (`assets/best.pt`) и дообучаем её на 7 классов.
Можно закрыть вкладку не больше чем на несколько минут: если Colab отключится, придётся начать с шага 2.

In [ ]:
!python -m train.train --data datasets/wastewise --base assets/best.pt --epochs 30 --imgsz 320 --out models/new_best.pt

## 7. Сравнить со старой моделью
Обе модели отвечают на **одни и те же** тестовые фото. Новая не должна стать хуже старой на 4 старых классах:
внизу будет ✅ (можно ставить на сайт) или ❌ (не ставить — что делать, смотрите `docs/TRAINING.md`).

In [ ]:
!python -m train.evaluate --test datasets/wastewise/test --old assets/best.pt --new models/new_best.pt --json models/compare.json

## 8. Скачать новую модель
Только если в шаге 7 стоит ✅. Скачается `new_best.pt` — дальше по `docs/TRAINING.md`, раздел «Поставить модель на сайт»
(переименовать в `best.pt` и загрузить в папку `assets/` репозитория).

In [ ]:
from google.colab import files
files.download("models/new_best.pt")
files.download("models/compare.json")